In [ ]:
import logging
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

logger = logging.getLogger("sae")
logger.setLevel(logging.INFO)


@torch.no_grad()
def set_decoder_norm_to_unit_norm_l2(
    W_dec: torch.nn.Parameter, input_dim: int, hidden_dim: int
) -> torch.Tensor:
    """
    Normalize decoder weight columns to unit norm.
    
    There's a major footgun here: we use this with both nn.Linear and nn.Parameter decoders.
    nn.Linear stores the decoder weights in a transposed format (input_dim, hidden_dim). 
    So, we pass the dimensions in to catch this error.
    """
    D, F = W_dec.shape
    assert D == input_dim, f"Expected input_dim={input_dim}, got {D}"
    assert F == hidden_dim, f"Expected hidden_dim={hidden_dim}, got {F}"
    
    eps = torch.finfo(W_dec.dtype).eps
    norm = torch.norm(W_dec.data, dim=0, keepdim=True)
    W_dec.data /= norm + eps
    return W_dec.data


def geometric_median_l2(x: torch.Tensor, max_iter: int = 100, tol: float = 1e-5) -> torch.Tensor:
    """
    Compute the geometric median of a set of points using Weiszfeld's algorithm.
    x: tensor of shape (N, D) where N is number of points, D is dimension.
    Returns: tensor of shape (D,)
    """
    y = x.mean(dim=0)
    for _ in range(max_iter):
        dists = (x - y).norm(dim=1, keepdim=True).clamp(min=1e-8)
        weights = 1.0 / dists
        y_new = (x * weights).sum(dim=0) / weights.sum()
        if (y_new - y).norm() < tol:
            break
        y = y_new
    return y


class SAE_l2w_encoder_decoder(nn.Module):
    """
    Sparse Autoencoder with untied weights (decoder is a separate nn.Linear) and L2 weight regularization.

    - Input is flattened to (N, D).
    - ReLU enforces non-negativity in the code (common for sparse coding).
    - Loss = MSE(recon, x) + l1 * mean(|z|) + l2_w * (||W_dec||_2 + ||W_enc||_2).
    - Initialization: Kaiming uniform for decoder, transpose-tied encoder, zero biases,
      decoder bias set to geometric median of data at training start.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        l1: float = 1e-3,
        l2_w: float = 1e-3,
        seed: int = 0,
    ) -> None:
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.l1 = float(l1)
        self.l2_w = float(l2_w)
        self.seed = int(seed)
        
        # Decoder: no bias in nn.Linear, we use separate b_dec
        self.decoder = nn.Linear(self.hidden_dim, self.input_dim, bias=False)
        
        # Encoder: has bias
        self.encoder = nn.Linear(self.input_dim, self.hidden_dim, bias=True)
        
        # Decoder bias (separate parameter, will be set to geometric median at training start)
        self.b_dec = nn.Parameter(torch.zeros(self.input_dim))

        # Initialize with reproducible seed
        gen = torch.Generator()
        gen.manual_seed(self.seed)
        
        # Kaiming uniform for decoder weights
        nn.init.kaiming_uniform_(self.decoder.weight, a=0.0, generator=gen)
        # Normalize decoder weights to unit norm
        self.decoder.weight.data = set_decoder_norm_to_unit_norm_l2(
            self.decoder.weight, self.input_dim, self.hidden_dim
        )
        
        # Encoder weights = transpose of decoder weights (tied init)
        self.encoder.weight.data = self.decoder.weight.T.clone()
        # Zero encoder bias
        self.encoder.bias.data.zero_()

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # Subtract decoder bias before encoding
        z = F.relu(self.encoder(x - self.b_dec))
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # Add decoder bias after decoding
        return self.decoder(z) + self.b_dec

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def loss(self, x: torch.Tensor, x_hat: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        recon = F.mse_loss(x_hat, x)
        sparsity = z.abs().mean()
        return recon + self.l1 * sparsity + self.l2_w * (self.decoder.weight.norm(p=2) + self.encoder.weight.norm(p=2))

    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

    def fit(
        self,
        loader: DataLoader,
        epochs: int = 50,
        lr: float = 1e-3,
        weight_decay: float = 0.0,
        device: Optional[torch.device] = None,
        log_interval: int = 100,
    ) -> None:
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)
        
        # Initialize b_dec to geometric median of first batch of data
        with torch.no_grad():
            first_batch = next(iter(loader))[0].to(device)
            self.b_dec.data = geometric_median_l2(first_batch)
        
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)

        step = 0
        self.train()  # use the standard nn.Module.train mode
        for epoch in range(1, epochs + 1):
            for xb, _ in loader:  # targets are xb itself (autoencoder)
                xb = xb.to(device, non_blocking=True)
                x_hat, z = self(xb)
                sparsity = (z != 0).sum().item()
                loss = self.loss(xb, x_hat, z)

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                
                # Enforce decoder norm to be 1 after each step
                self.decoder.weight.data = set_decoder_norm_to_unit_norm_l2(
                    self.decoder.weight, self.input_dim, self.hidden_dim
                )

                if step % log_interval == 0:
                    print(f"epoch={epoch} step={step} loss={loss.item():.6f} sparsity={sparsity}")
                step += 1

In [ ]:
import logging
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

logger = logging.getLogger("sae")
logger.setLevel(logging.INFO)


@torch.no_grad()
def set_decoder_norm_to_unit_norm(
    W_dec: torch.nn.Parameter, input_dim: int, hidden_dim: int
) -> torch.Tensor:
    """
    Normalize decoder weight columns to unit norm.
    
    There's a major footgun here: we use this with both nn.Linear and nn.Parameter decoders.
    nn.Linear stores the decoder weights in a transposed format (input_dim, hidden_dim). 
    So, we pass the dimensions in to catch this error.
    """
    D, F = W_dec.shape
    assert D == input_dim, f"Expected input_dim={input_dim}, got {D}"
    assert F == hidden_dim, f"Expected hidden_dim={hidden_dim}, got {F}"
    
    eps = torch.finfo(W_dec.dtype).eps
    norm = torch.norm(W_dec.data, dim=0, keepdim=True)
    W_dec.data /= norm + eps
    return W_dec.data


def geometric_median(x: torch.Tensor, max_iter: int = 100, tol: float = 1e-5) -> torch.Tensor:
    """
    Compute the geometric median of a set of points using Weiszfeld's algorithm.
    x: tensor of shape (N, D) where N is number of points, D is dimension.
    Returns: tensor of shape (D,)
    """
    # Initialize with the mean
    y = x.mean(dim=0)
    for _ in range(max_iter):
        # Compute distances from current estimate
        dists = (x - y).norm(dim=1, keepdim=True).clamp(min=1e-8)
        # Weighted average
        weights = 1.0 / dists
        y_new = (x * weights).sum(dim=0) / weights.sum()
        # Check convergence
        if (y_new - y).norm() < tol:
            break
        y = y_new
    return y


class SAE_l1w_encoder_decoder(nn.Module):
    """
    Sparse Autoencoder with untied weights (decoder is a separate nn.Linear) and L1 sparsity on activations.

    - Input is flattened to (N, D).
    - ReLU enforces non-negativity in the code (common for sparse coding).
    - Loss = MSE(recon, x) + l1 * mean(|z|).
    - Initialization: Kaiming uniform for decoder, transpose-tied encoder, zero biases,
      decoder bias set to geometric median of data at training start.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        l1: float = 1e-3,
        l1_w: float = 1e-3,
        seed: int = 0,
    ) -> None:
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.l1 = float(l1)
        self.l1_w = float(l1_w)
        self.seed = int(seed)
        
        # Decoder: no bias in nn.Linear, we use separate b_dec
        self.decoder = nn.Linear(self.hidden_dim, self.input_dim, bias=False)
        
        # Encoder: has bias
        self.encoder = nn.Linear(self.input_dim, self.hidden_dim, bias=True)
        
        # Decoder bias (separate parameter, will be set to geometric median at training start)
        self.b_dec = nn.Parameter(torch.zeros(self.input_dim))

        # Initialize with reproducible seed
        gen = torch.Generator()
        gen.manual_seed(self.seed)
        
        # Kaiming uniform for decoder weights
        nn.init.kaiming_uniform_(self.decoder.weight, a=0.0, generator=gen)
        # Normalize decoder weights to unit norm
        self.decoder.weight.data = set_decoder_norm_to_unit_norm(
            self.decoder.weight, self.input_dim, self.hidden_dim
        )
        
        # Encoder weights = transpose of decoder weights (tied init)
        self.encoder.weight.data = self.decoder.weight.T.clone()
        # Zero encoder bias
        self.encoder.bias.data.zero_()

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # Subtract decoder bias before encoding
        z = F.relu(self.encoder(x - self.b_dec))
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # Add decoder bias after decoding
        return self.decoder(z) + self.b_dec

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def loss(self, x: torch.Tensor, x_hat: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        recon = F.mse_loss(x_hat, x)
        sparsity = z.abs().mean()
        return recon + self.l1 * sparsity + self.l1_w * (self.decoder.weight.abs().mean() + self.encoder.weight.abs().mean())

    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

    def fit(
        self,
        loader: DataLoader,
        epochs: int = 50,
        lr: float = 1e-3,
        weight_decay: float = 0.0,
        device: Optional[torch.device] = None,
        log_interval: int = 100,
    ) -> None:
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)
        
        # Initialize b_dec to geometric median of first batch of data
        with torch.no_grad():
            first_batch = next(iter(loader))[0].to(device)
            self.b_dec.data = geometric_median(first_batch)
        
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)

        step = 0
        self.train()  # use the standard nn.Module.train mode
        for epoch in range(1, epochs + 1):
            for xb, _ in loader:  # targets are xb itself (autoencoder)
                xb = xb.to(device, non_blocking=True)
                x_hat, z = self(xb)
                sparsity = (z != 0).sum().item()
                loss = self.loss(xb, x_hat, z)

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                
                # Enforce decoder norm to be 1 after each step
                self.decoder.weight.data = set_decoder_norm_to_unit_norm(
                    self.decoder.weight, self.input_dim, self.hidden_dim
                )

                if step % log_interval == 0:
                    print(f"epoch={epoch} step={step} loss={loss.item():.6f} sparsity={sparsity}")
                step += 1

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

# Load MNIST data as input and target (unsupervised)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),  # gives [0,1] float32, shape=[1,28,28]
    transforms.Lambda(lambda x: x.view(-1)),  # flatten to [784]
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)

def ae_collate(batch):
    xs = torch.stack([x for x, _ in batch])
    return xs, xs

loader = DataLoader(
    mnist_train,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=ae_collate,
)

input_dim = 28 * 28

# Make output directory for saving models
os.makedirs("trained_autoencoder_MNISTs", exist_ok=True)

# Hyperparameter sweep for l1_w and l1
l1_values = [1e-1]
l2_w_values = [0, 1e-5]
seeds = [0, 1, 2]
autoencoders = {}


for seed in seeds:
    for l1 in l1_values:
        for l2_w in l2_w_values:
            # unique model name for each sweep setting
            model_name = f"sae_l1w_l1_{l1}_l2w_{l2_w}_100ep_tied_init_seed_{seed}"
            ae = SAE_l2w_encoder_decoder(
                seed=seed,
                input_dim=input_dim, 
                hidden_dim=2 * input_dim, 
                l1=l1, 
                l2_w=l2_w,
            )
            print(f"Training {model_name}")
            ae.fit(
                loader=loader,
                epochs=100,
                lr=1e-3,
                weight_decay=0.0,
                log_interval=100,
            )
            autoencoders[model_name] = ae

            # Save the trained model
            save_path = os.path.join("trained_autoencoder_MNISTs", f"{model_name}.pt")
            torch.save(ae, save_path)
            print(f"Saved model to {save_path}")



In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

# Load MNIST data as input and target (unsupervised)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),  # gives [0,1] float32, shape=[1,28,28]
    transforms.Lambda(lambda x: x.view(-1)),  # flatten to [784]
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)

def ae_collate(batch):
    xs = torch.stack([x for x, _ in batch])
    return xs, xs

loader = DataLoader(
    mnist_train,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=ae_collate,
)

input_dim = 28 * 28

# Make output directory for saving models
os.makedirs("trained_autoencoder_MNISTs", exist_ok=True)

# Hyperparameter sweep for l1_w and l1
l1_values = [1e-1]
l1_w_values = [0, 1e-3]
seeds = [0, 1, 2]
autoencoders = {}


for seed in seeds:
    for l1 in l1_values:
        for l1_w in l1_w_values:
            # unique model name for each sweep setting
            model_name = f"sae_l1w_l1_{l1}_l1w_{l1_w}_100ep_tied_init_seed_{seed}"
            ae = SAE_l1w_encoder_decoder(
                seed=seed,
                input_dim=input_dim, 
                hidden_dim=2 * input_dim, 
                l1=l1, 
                l1_w=l1_w,
            )
            print(f"Training {model_name}")
            ae.fit(
                loader=loader,
                epochs=100,
                lr=1e-3,
                weight_decay=0.0,
                log_interval=100,
            )
            autoencoders[model_name] = ae

            # Save the trained model
            save_path = os.path.join("trained_autoencoder_MNISTs", f"{model_name}.pt")
            torch.save(ae, save_path)
            print(f"Saved model to {save_path}")

